# Day 7: 施策評価 (Before/After比較)

## 学習目標
- 「施策あり/なし」を比較する評価の型（Before/After比較）を身につける
- `congestion_pricing`（時間帯別の課金関数）をリンクに設定し、料金施策の効果を評価する
- 車線数の変更（車線閉鎖・拡幅）や信号の青時間配分変更を、同じ型で評価する
- 総旅行時間・平均遅れなど、複数の指標を1つの比較表にまとめる


In [ ]:
from uxsim import World
import pandas as pd

def build_base_world(toll_link=False, lanes_mainline=1, seed=0):
    W = World(
        name="",
        deltan=5,
        tmax=4000,
        print_mode=0, save_mode=0, show_mode=0,
        random_seed=seed,
    )
    W.addNode("orig", 0, 0)
    W.addNode("mid", 1, 0)
    W.addNode("dest", 2, 0)
    W.addNode("bypass1", 1, 1)

    def toll(t):
        # ピーク時間帯(1000-2500s)だけ課金し、それ以外は0
        return 50.0 if 1000 <= t <= 2500 else 0.0

    W.addLink("main_up", "orig", "mid", length=2000, free_flow_speed=20,
              number_of_lanes=lanes_mainline,
              congestion_pricing=toll if toll_link else None)
    W.addLink("main_down", "mid", "dest", length=2000, free_flow_speed=20, number_of_lanes=lanes_mainline)
    # 迂回路(遠回りだが課金なし)
    W.addLink("bypass_up", "orig", "bypass1", length=3000, free_flow_speed=20, number_of_lanes=1)
    W.addLink("bypass_down", "bypass1", "dest", length=3000, free_flow_speed=20, number_of_lanes=1)

    W.adddemand("orig", "dest", t_start=0, t_end=3000, flow=0.5)
    return W

def summarize(W, label):
    a = W.analyzer
    return {
        "scenario": label,
        "total_travel_time_s": a.total_travel_time,
        "average_travel_time_s": a.average_travel_time,
        "average_delay_s": a.average_delay,
        "trip_completed": a.trip_completed,
    }

results = []

W_before = build_base_world(toll_link=False)
W_before.exec_simulation()
results.append(summarize(W_before, "Before (課金なし)"))

W_after = build_base_world(toll_link=True)
W_after.exec_simulation()
results.append(summarize(W_after, "After (ピーク時課金あり)"))

pd.DataFrame(results)


## Part B: 演習

1. 上のBefore/Afterに加えて、**車線閉鎖シナリオ**（`lanes_mainline=1` → 事故等で
   一時的に容量が下がったケースを `capacity_out` を絞るなどで再現）を追加し、
   3シナリオの比較表を作ってください。
2. 課金額（`toll`関数の `50.0`）や課金時間帯を変えて、「総旅行時間が最小になる課金設定」を
   探索してください（すべて迂回されすぎても、すべて素通りされても最適ではないはずです）。
3. `basic_to_pandas()` を使って、評価指標をまとめて取得する関数に置き換えてみましょう。


In [ ]:
# TODO 1: 車線閉鎖シナリオを追加する
# W_incident = build_base_world(toll_link=False, lanes_mainline=1)
# (例えば main_up の capacity_out を意図的に下げる、または number_of_lanes を変えて再構築する)

# TODO 2: 課金額を変えたときの総旅行時間の変化を調べる
# for toll_value in [0, 20, 50, 100]:
#     ...


## Part C: 考察

- 「総旅行時間の最小化」だけを評価指標にすると見落としてしまう観点（公平性、
  課金による収入の使い道、迂回路周辺の生活道路への影響など）を挙げてください。
- 自分がもし国交省・高速道路会社に施策の効果を説明するとしたら、
  この比較表に加えてどんな情報を添えるべきか考えてみてください。
